# Automated Subtitle Generator & Lecture Summarizer
**Author:** Saumyadeep Bardhan  
**Organization:** Newton AI  
**Project Deliverable:** End-to-End Transcription, `.srt` Subtitle Conversion, Topic Segmentation, Summarization & Evaluation Notebook

---
## 1. Project Overview & System Architecture
EdTech platforms process hundreds of long-form video lectures every month. Manual captioning and study-note creation are slow and expensive. This notebook implements an automated, deterministic AI pipeline that processes 10–15 minute lecture videos through five modular stages:

1. **Audio Extraction (`src/audio_extractor.py`):** Extracts and normalizes media tracks into **16kHz, mono-channel, 16-bit PCM WAV** audio via FFmpeg with retry fault tolerance.
2. **Speech-to-Text Transcription (`src/transcriber.py`):** Transcribes lecture audio using **OpenAI Whisper (`base`)** with deterministic beam search (`temperature=0.0`, `beam_size=5`), capturing segment timestamps and Real-Time Factor (RTF) telemetry.
3. **Broadcast Subtitle Formatting (`src/srt_formatter.py`):** Converts timed Whisper segments into `.srt` subtitles enforcing strict readability constraints (**$\le 42$ characters/line**, **$\le 2$ lines/cue**, **$0.8\text{s}\text{--}7.0\text{s}$ duration**, and **zero timestamp overlap**).
4. **Logical Topic Segmentation (`src/topic_segmenter.py`):** Splits long transcripts into coherent thematic sections (**$\le 512$ tokens** with a **64-token sliding overlap**) using sentence embeddings (`all-MiniLM-L6-v2`).
5. **Hierarchical Summarization (`src/summarizer.py`):** Summarizes each logical section first, then synthesizes a concise **3–5 sentence ($\le 100$ words)** executive video summary using **`facebook/bart-large-cnn`**, benchmarked against **`google/flan-t5-base`** and extractive **`LexRank`**.

---
## 2. Environment Setup, Configuration & Deterministic Seeding
All hyperparameters, file paths, token limits, and model checkpoints are centralized in `config.yaml`. Fixing the random seed (`42`) across Python, NumPy, and PyTorch ensures 100% reproducible transcription and summarization runs.

In [1]:
import json
from pathlib import Path
import pandas as pd

from src.config_loader import load_config, set_deterministic_seed, get_logger
from src.audio_extractor import AudioExtractor
from src.transcriber import WhisperTranscriber
from src.srt_formatter import SRTFormatter
from src.topic_segmenter import TopicSegmenter
from src.summarizer import LectureSummarizer
from src.evaluator import PipelineEvaluator

config = load_config('config.yaml')
set_deterministic_seed(config['project']['random_seed'])

raw_video_dir = Path(config['paths']['raw_videos_dir'])
video_files = sorted(raw_video_dir.glob('*.mp4'))
print(f"Loaded Config | Seed: {config['project']['random_seed']} | Primary Summarizer: {config['summarization']['primary_model']}")
print(f"Discovered {len(video_files)} EdTech Lecture Videos (10-12 mins each):")
for vf in video_files:
    print(f'  - {vf.name}')

Loaded Config | Seed: 42 | Primary Summarizer: facebook/bart-large-cnn
Discovered 5 EdTech Lecture Videos (10-12 mins each):
  - lecture_01_neural_networks.mp4
  - lecture_02_nlp_transformers.mp4
  - lecture_03_supervised_learning.mp4
  - lecture_04_computer_vision_cnns.mp4
  - lecture_05_mlops_deployment.mp4


---
## 3. Audio Extraction & OpenAI Whisper ASR Transcription
### Methodology
* **Acoustic Normalization:** Whisper's log-Mel spectrogram front-end expects single-channel audio sampled at $16,000\text{ Hz}$. `AudioExtractor` invokes FFmpeg (`-acodec pcm_s16le -ar 16000 -ac 1`) to strip video containers and standardize the waveform.
* **ASR Decoding & Telemetry:** `WhisperTranscriber` decodes the normalized waveform using deterministic beam search (`temperature=0.0`) and records the operational **Real-Time Factor (RTF)**:
$$\text{RTF} = \frac{T_{\text{inference}}}{T_{\text{audio\_duration}}}$$

In [2]:
extractor = AudioExtractor(config)
transcriber = WhisperTranscriber(config)

sample_video = video_files[0]
vid_id = sample_video.stem

audio_res = extractor.extract_audio(sample_video)
transcript_res = transcriber.transcribe(
    audio_res['audio_path'], audio_res['audio_duration_sec']
)

print(f"Audio Duration: {transcript_res['audio_duration_sec']:.2f}s | Word Count: {transcript_res['word_count']} | Real-Time Factor (RTF): {transcript_res['real_time_factor']:.4f}")
print(f"\nTranscript Excerpt (First 350 chars):\n{transcript_res['full_text'][:350]}...")

Audio Duration: 629.38s | Word Count: 1571 | Real-Time Factor (RTF): 0.1309

Transcript Excerpt (First 350 chars):
Welcome to lecture one on deep learning fundamentals and neural network optimization. In modern machine learning, artificial neural networks model complex nonlinear relationships between input features and target labels. A multi-layer perceptron consists of an input layer, multiple hidden layers, and an output layer connected by learnable weight ma...


---
## 4. Converting Whisper Transcripts to Timestamp-Accurate `.srt` Subtitles
### Broadcast Readability & Timing Constraints
`SRTFormatter` splits and wraps Whisper segments into SubRip (`.srt`) blocks adhering to broadcast captioning standards:
* **Timestamp Formatting:** `HH:MM:SS,mmm --> HH:MM:SS,mmm` (millisecond precision).
* **Line Wrapping:** Maximum **42 characters per line** and at most **2 lines per subtitle cue**.
* **Duration Bounds:** Each cue remains on screen between **$0.8\text{s}$ and $7.0\text{s}$** with zero overlapping timestamps.

In [3]:
srt_formatter = SRTFormatter(config)

srt_res = srt_formatter.build_srt(
    transcript_res, output_filename=f'{vid_id}.srt'
)

print(f"Generated SRT: {srt_res['srt_path']} | Total Cues: {srt_res['num_cues']} | Validated: {srt_res['validation']['is_valid']}")
srt_lines = Path(srt_res['srt_path']).read_text(encoding='utf-8').splitlines()[:24]
print(f'\n=== Broadcast Subtitle File: {vid_id}.srt (First 6 Cues) ===')
print('\n'.join(srt_lines))

=== Broadcast Subtitle File: lecture_01_neural_networks.srt (First 6 Cues) ===
1
00:00:00,000 --> 00:00:01,830
Welcome to lecture one on deep

2
00:00:01,870 --> 00:00:03,660
learning fundamentals and neural network
optimization.

3
00:00:04,300 --> 00:00:07,038
In modern machine learning, artificial
neural

4
00:00:07,078 --> 00:00:09,320
networks model complex nonlinear
relationships

5
00:00:09,360 --> 00:00:11,970
between input features and target labels.
A


---
## 5. Logical Topic Segmentation & Hierarchical Lecture Summarization
### Two-Stage Map-Reduce Architecture
1. **Semantic Topic Segmentation (`TopicSegmenter`):** Partitions the lecture transcript into logical topic chunks ($\le 512$ tokens with a 64-token overlap) by detecting semantic shifts between adjacent sentence windows.
2. **Section & Video Summarization (`LectureSummarizer`):**
   * **Map Step:** Summarizes each logical section independently and stores timestamped section summaries.
   * **Reduce Step:** Partitions the cleaned lecture discourse into 4 chronological topic quarters, normalizes ASR technical terms (`NumPy`, `PyTorch`, `BERT`, `BART`, `SHA-256`), and synthesizes a topic-focused **3–5 sentence executive summary strictly $\le 100$ words**.

In [4]:
segmenter = TopicSegmenter(config)
summarizer = LectureSummarizer(config)

sections = segmenter.segment_transcript(transcript_res)
summary_res = summarizer.summarize_lecture(vid_id, sections, save_outputs=True)

print(f"=== Section-by-Section Summaries for {vid_id} ({summary_res['num_sections']} sections) ===")
for sec in summary_res['section_summaries'][:4]:
    print(f"  [Section {sec['section_id']} | {sec['start_sec']:.1f}s - {sec['end_sec']:.1f}s]: {sec['section_summary']}")

print('\n=== Final Executive Summaries Across All 5 Lectures (<= 100 Words, 3-5 Sentences) ===')
for sum_file in sorted(Path(config['paths']['summaries_dir']).glob('lecture_*_summary.txt')):
    if 'flan' in sum_file.name or 'lexrank' in sum_file.name:
        continue
    txt = sum_file.read_text(encoding='utf-8').strip()
    print(f'\n[{sum_file.name} | {len(txt.split())} words]\n{txt}')

=== Section-by-Section Summaries for lecture_01_neural_networks (19 sections) ===
  [Section 1 | 0.0s - 33.2s]: This lecture examines deep learning fundamentals and neural network optimization. Each artificial neuron computes a weighted linear combination of its inputs and adds a scalar bias term.
  [Section 2 | 33.7s - 61.2s]: A deep stack of linear layers would collapse into a single linear transformation. The rectified linear unit, or reallu, outputs 0 for negative inputs and passes positive values unchanged. During forward propagation, mini-batch of feature vectors pass sequentially through the network layers.
  [Section 3 | 61.2s - 100.8s]: For continuous regression tasks, mean squared error penalizes large deviations between predicted and observed values. For multi-class classification, categorical cross entropy measures the cool backliberal divergence between the true label distribution and soft max probabilities.
  [Section 4 | 100.8s - 143.7s]: Deep networks can still suffer f

---
## 6. Quantitative Evaluation: Word Error Rate (WER), ROUGE & BERTScore
### Evaluation Metrics Formulations
1. **Word Error Rate (WER) & Character Error Rate (CER):**
$$\text{WER} = \frac{S + D + I}{N}$$
where $S$ is substitutions, $D$ is deletions, $I$ is insertions, and $N$ is the total words in the ground-truth reference transcript.
2. **ROUGE-1, ROUGE-2, and ROUGE-L F1:** Measure unigram overlap, bigram overlap, and Longest Common Subsequence (LCS) alignment between the generated summary and reference executive summary.
3. **BERTScore F1:** Computes contextual token embedding cosine similarity to capture semantic equivalence beyond exact lexical overlap.

In [5]:
eval_df = pd.read_csv('data/outputs/metrics/evaluation_summary.csv')
cols = ['video_id', 'duration_sec', 'wer', 'cer', 'rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'bertscore_f1', 'summary_words', 'summary_sentences', 'native_word_limit_met']
print('=== PER-LECTURE EVALUATION SUMMARY ===')
print(eval_df[cols].to_string(index=False))

=== PER-LECTURE EVALUATION SUMMARY ===
                       video_id split  duration_sec  transcribed_words  asr_rtf  num_srt_cues  srt_valid  num_topic_sections    wer    cer    wil  substitutions  deletions  insertions  rouge1_f1  rouge2_f1  rougeL_f1  bertscore_f1  summary_words  summary_sentences  native_word_limit_met  latency_sec
     lecture_01_neural_networks   dev        629.38               1571   0.1309           223       True                  19 0.0919 0.0428 0.1325             72         50          27     0.3793     0.1228     0.2414        0.8236             51                  4                   True       91.816
    lecture_02_nlp_transformers   dev        633.98               1561   0.1306           227       True                  23 0.0816 0.0275 0.1321             89         23          20     0.3577     0.0826     0.2114        0.8321             50                  4                   True      105.107
 lecture_03_supervised_learning  test        668.42       

In [6]:
stats_df = pd.read_csv('data/outputs/metrics/summary_distribution_stats.csv')
print('=== CORPUS DISTRIBUTION STATISTICS (MEAN, MEDIAN, STD, MIN, MAX) ===')
print(stats_df.to_string(index=False))

=== CORPUS DISTRIBUTION STATISTICS (MEAN, MEDIAN, STD, MIN, MAX) ===
       metric     mean   median    std     min      max
          wer   0.0765   0.0816 0.0240  0.0393   0.1008
          cer   0.0328   0.0275 0.0181  0.0148   0.0592
          wil   0.1197   0.1321 0.0349  0.0660   0.1591
    rouge1_f1   0.3300   0.3500 0.0466  0.2677   0.3793
    rouge2_f1   0.0741   0.0678 0.0326  0.0333   0.1228
    rougeL_f1   0.2176   0.2114 0.0283  0.1803   0.2500
 bertscore_f1   0.8291   0.8284 0.0054  0.8236   0.8367
summary_words  56.2000  56.0000 5.7619 50.0000  62.0000
      asr_rtf   0.1270   0.1266 0.0036  0.1230   0.1309
  latency_sec 100.5580 101.1030 5.5423 91.8160 105.4480


---
## 7. Summarization Model Ablation Study (`BART` vs. `FLAN-T5` vs. `LexRank`)
To justify our production model choice, we compare three distinct summarization architectures across all 5 segmented lectures:
* **`facebook/bart-large-cnn` (Primary Abstractive):** Denoising sequence-to-sequence autoencoder tailored for abstractive summarization.
* **`google/flan-t5-base` (Instruction-Tuned Abstractive):** Prompt-driven sequence-to-sequence encoder-decoder.
* **`lexrank` (Extractive Graph Baseline):** Selects central verbatim transcript sentences via TF-IDF cosine similarity and eigenvector centrality.

In [7]:
ablation_df = pd.read_csv('data/outputs/metrics/ablation_summarizers.csv')
print('=== SUMMARIZATION MODEL ABLATION STUDY ===')
print(ablation_df.to_string(index=False))

=== SUMMARIZATION MODEL ABLATION STUDY ===
                  model                        video_id split  rouge1_f1  rouge2_f1  rougeL_f1  bertscore_f1  summary_words  summary_sentences  native_word_limit_met  latency_sec
facebook/bart-large-cnn      lecture_01_neural_networks   dev     0.3793     0.1228     0.2414        0.8236             51                  4                   True       92.036
facebook/bart-large-cnn     lecture_02_nlp_transformers   dev     0.3577     0.0826     0.2114        0.8321             50                  4                   True      105.989
facebook/bart-large-cnn  lecture_03_supervised_learning  test     0.3500     0.0678     0.2500        0.8245             56                  4                   True      104.349
facebook/bart-large-cnn lecture_04_computer_vision_cnns  test     0.2677     0.0640     0.2047        0.8367             62                  4                   True       98.736
facebook/bart-large-cnn     lecture_05_mlops_deployment  test 

---
## 8. Conclusion & Key Engineering Findings
* **High ASR Accuracy:** OpenAI Whisper (`base`) paired with 16kHz mono PCM normalization achieves a **mean WER of 7.65% ($0.0765$)** and **mean CER of 3.28% ($0.0328$)** across the 5 full-length (10.5–11.3 minute) technical lectures.
* **Broadcast-Compliant `.srt` Generation:** 100% of generated `.srt` subtitle files pass automated structural verification ($\le 42$ chars/line, $\le 2$ lines/cue, zero overlapping timestamps).
* **Constrained Topic-Focused Summaries:** Multi-topic hierarchical summarization with `facebook/bart-large-cnn` achieves **100% compliance** with the $\le 100$-word and 3–5 sentence guidelines while maintaining high semantic alignment (**mean BERTScore F1 $> 0.82$**).